# Transformers — One‑Stop Math + Deep Theory + End‑to‑End Worked Example (PyTorch)

This notebook is your end-to-end recap of the Transformer block:
- embeddings + positions (conceptually)
- self-attention (Q, K, V) **with detailed explanation**
- multi-head attention (MHA)
- residual connections + LayerNorm
- feedforward network (FFN)
- a tiny **numeric** Transformer block you can trace
- plus a PyTorch `nn.MultiheadAttention` API demo

We keep the math in paper notation and directly map to code.

## 0) Big picture: what Transformers changed vs RNNs

RNNs are “deep in time”: to connect token 1 to token \(T\), information must pass through \(T-1\) recurrent steps.

Transformers remove this sequential dependency inside a layer:
- Every token can interact with every other token **in one attention operation**.
- The model is “deep in layers” but not forced to be deep in time for dependency modeling.

This is one core reason Transformers handle long-range dependencies better.

## 1) Input representation

In language models, each token index is mapped to an embedding vector (learned lookup).

Let:
- sequence length \(T\)
- model width \(d_{model}\)

After embedding + positional encoding:
$$
Z_0 \in \mathbb{R}^{T \times d_{model}}.
$$

In this notebook we will *directly define* a small numeric \(Z_0\) so you can trace computations.

## 2) Scaled dot-product self-attention (core math + explanation)

Given \(Z \in \mathbb{R}^{T \times d_{model}}\), we compute three projections:

$$
Q = ZW_Q,\quad K = ZW_K,\quad V = ZW_V
$$

Where typically:
$$
W_Q, W_K, W_V \in \mathbb{R}^{d_{model} \times d_k}.
$$

### 2.1 Why Q, K, V?
- **Query (Q)**: what this token is looking for
- **Key (K)**: what each token offers (address/label)
- **Value (V)**: the information/content to retrieve

### 2.2 Attention scores
$$
S = \frac{QK^\top}{\sqrt{d_k}} \in \mathbb{R}^{T \times T}
$$

- \(S_{ij}\) is the similarity between query of token \(i\) and key of token \(j\).
- Divide by \(\sqrt{d_k}\) to keep magnitudes stable as \(d_k\) grows.

### 2.3 Softmax turns scores into weights
$$
A = \mathrm{softmax}(S)
$$
Softmax is applied **row-wise**:
- each row corresponds to one query token \(i\)
- the row sums to 1 (distribution over keys)

### 2.4 Weighted sum of values
$$
\mathrm{Attn}(Q,K,V) = AV
$$

So each token output is a mixture of value vectors, weighted by attention.

## 3) Multi-head attention (MHA): why multiple heads?

One head might focus on one type of relation (e.g., nearby syntax). Another head might focus on long-range semantics.

For head \(r\):
$$
Q^{(r)} = ZW_Q^{(r)},\quad K^{(r)} = ZW_K^{(r)},\quad V^{(r)} = ZW_V^{(r)}
$$
$$
\text{head}_r = \mathrm{Attn}(Q^{(r)},K^{(r)},V^{(r)})
$$

Concatenate heads and project back:
$$
\mathrm{MHA}(Z) = \mathrm{Concat}(\text{head}_1,\dots,\text{head}_h)W_O
$$

Shape intuition (common configuration):
- \(Z\): \(T \times d_{model}\)
- each head output: \(T \times d_v\)
- concat: \(T \times (h d_v)\)
- \(W_O\): \((h d_v) \times d_{model}\)
- output: \(T \times d_{model}\)

## 4) Residual connections, LayerNorm, and FFN

### 4.1 Residual + LayerNorm
After attention:
$$
Z_1 = \mathrm{LayerNorm}(Z_0 + \mathrm{MHA}(Z_0))
$$

**Why residual matters**
- It creates a direct gradient path that helps optimization.
- It lets the network learn “small corrections” rather than rewriting representations.

### 4.2 Position-wise FFN
A 2-layer MLP applied independently to each token (same weights per position):
$$
\mathrm{FFN}(Z_1) = \phi(Z_1W_1 + b_1)W_2 + b_2
$$
where \(\phi\) is typically ReLU or GELU.

Then second residual:
$$
Z_2 = \mathrm{LayerNorm}(Z_1 + \mathrm{FFN}(Z_1))
$$

## 5) Worked numeric example (tiny Transformer block in PyTorch)

We set:
- \(T=2\)
- \(d_{model}=4\)
- \(h=2\) heads
- \(d_k=d_v=2\)

We define weights explicitly and print:
- Q, K, V for each head
- scores \(S\), weights \(A\)
- head outputs
- concatenation + output projection
- residual + LayerNorm
- FFN + residual + LayerNorm

This is designed to be traceable by hand.

In [ ]:
import torch
import torch.nn.functional as F
import math

torch.set_printoptions(precision=4, sci_mode=False)

T = 2
d_model = 4
h = 2
d_k = d_v = d_model // h  # 2

# Input Z0 (T x d_model)
Z0 = torch.tensor([
    [1.0, 0.5, -0.5, 2.0],
    [0.0, 1.0,  1.5, -1.0],
])

# Per-head projections (d_model x d_k)
WQ1 = torch.tensor([[0.2, 0.0],[0.1, 0.3],[0.0, 0.2],[0.1,-0.1]])
WK1 = torch.tensor([[0.1, 0.2],[0.0, 0.1],[0.3, 0.0],[-0.2,0.2]])
WV1 = torch.tensor([[0.0, 0.2],[0.1, 0.0],[0.2, 0.1],[0.0,0.3]])

WQ2 = torch.tensor([[0.0, 0.1],[0.2, 0.0],[0.1, 0.2],[0.3,-0.1]])
WK2 = torch.tensor([[0.2, 0.0],[0.1, 0.1],[0.0, 0.2],[-0.1,0.3]])
WV2 = torch.tensor([[0.1, 0.0],[0.0, 0.2],[0.2, 0.0],[0.1,0.1]])

# Output projection (h*d_v x d_model)
WO = torch.tensor([[0.2, 0.0, 0.1, 0.0],
                   [0.0, 0.3, 0.0, 0.1],
                   [0.1, 0.0, 0.2, 0.0],
                   [0.0, 0.1, 0.0, 0.2]])

def attention(Z, WQ, WK, WV):
    Q = Z @ WQ  # (T, d_k)
    K = Z @ WK  # (T, d_k)
    V = Z @ WV  # (T, d_v)
    scores = (Q @ K.T) / math.sqrt(d_k)   # (T, T)
    A = F.softmax(scores, dim=1)          # row-wise softmax over keys
    out = A @ V                           # (T, d_v)
    return Q, K, V, scores, A, out

Q1, K1, V1, S1, A1, H1 = attention(Z0, WQ1, WK1, WV1)
Q2, K2, V2, S2, A2, H2 = attention(Z0, WQ2, WK2, WV2)

print("=== Head 1 ===")
print("Q1:\n", Q1)
print("K1:\n", K1)
print("V1:\n", V1)
print("Scores S1:\n", S1)
print("Weights A1:\n", A1)
print("Output H1:\n", H1)

print("\n=== Head 2 ===")
print("Q2:\n", Q2)
print("K2:\n", K2)
print("V2:\n", V2)
print("Scores S2:\n", S2)
print("Weights A2:\n", A2)
print("Output H2:\n", H2)

Hcat = torch.cat([H1, H2], dim=1)   # (T, 4)
MHA  = Hcat @ WO                    # (T, d_model)

print("\nConcatenated heads Hcat:\n", Hcat)
print("\nMHA(Z0):\n", MHA)

In [ ]:
# Residual + LayerNorm after attention
Z1 = F.layer_norm(Z0 + MHA, normalized_shape=(d_model,))
print("Z1 = LayerNorm(Z0 + MHA):\n", Z1)

# FFN
d_ff = 6
W1 = torch.tensor([[0.2, 0.0, 0.1, 0.0, 0.1, -0.1],
                   [0.0, 0.3, 0.0, 0.1, 0.0,  0.2],
                   [0.1, 0.0, 0.2, 0.0, 0.1,  0.0],
                   [0.0, 0.1, 0.0, 0.2, 0.0,  0.1]])  # (4x6)
b1 = torch.zeros((d_ff,))
W2 = torch.tensor([[0.1, 0.0, 0.2, 0.0],
                   [0.0, 0.2, 0.0, 0.1],
                   [0.2, 0.0, 0.1, 0.0],
                   [0.0, 0.1, 0.0, 0.2],
                   [0.1, 0.0, 0.0, 0.1],
                   [0.0, 0.1, 0.2, 0.0]])  # (6x4)
b2 = torch.zeros((d_model,))

FF1 = F.relu(Z1 @ W1 + b1)      # (T, d_ff)
FF2 = FF1 @ W2 + b2             # (T, d_model)

print("\nFF1 (after ReLU):\n", FF1)
print("\nFF2 (projected back):\n", FF2)

Z2 = F.layer_norm(Z1 + FF2, normalized_shape=(d_model,))
print("\nZ2 = LayerNorm(Z1 + FFN(Z1)):\n", Z2)

## 6) Mapping to PyTorch `nn.MultiheadAttention` (API learning)

PyTorch’s built-in module handles:
- projection to Q/K/V
- multi-head split/merge
- attention softmax
- output projection

Default input shape is `(seq_len, batch, embed_dim)`. We'll use `batch=1`.

We set simple weights (not matching the manual 2-head exactly) to illustrate how the API works.

In [ ]:
from torch import nn

mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=h, bias=False, batch_first=False)

with torch.no_grad():
    # Use simple deterministic weights (scaled identity) for Q,K,V
    WQ = torch.eye(d_model) * 0.5
    WK = torch.eye(d_model) * 0.5
    WV = torch.eye(d_model) * 0.5
    mha.in_proj_weight.copy_(torch.cat([WQ, WK, WV], dim=0))
    mha.out_proj.weight.copy_(torch.eye(d_model))

Z = Z0.unsqueeze(1)  # (T,1,d_model)
attn_out, attn_weights = mha(Z, Z, Z, need_weights=True, average_attn_weights=False)

print("attn_out (T,1,d_model):\n", attn_out.squeeze(1))
print("\nattn_weights (batch, heads, T, T):\n", attn_weights)

## 7) Practical reading cues (how to read Transformer math in papers)

When you see a new Transformer variant in a paper, do this:
1) Write down shapes for \(Z, Q, K, V, S, A\). (This prevents 80% of confusion.)
2) Locate where softmax happens (over which axis).
3) Identify residual paths and norms (pre-norm vs post-norm).
4) Identify what changed: attention form? FFN? normalization? positional encoding?

If you can do (1)-(3), you can read most Transformer papers productively.